# 🔄 Chapter 6: Continuous Integration and Continuous Deployment

📖 Book: Chapter 6, pp. 236–262 · ⏱️ 2–3 hours · 🧱 Needs: Ch 3, 5 · Level: Intermediate

---

> **🎯 Key idea:** Fail-fast CI gates verify syntax, tests, and artifacts on every commit so only immutable, validated builds reach production.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** Jenkins + `rsync` (NFSOPS) / CodePipeline  
> **🆕 Now (2026):** GitHub Actions (`actions/setup-python`, OIDC)  
> **Why:** Zero-server CI with built-in caching & matrix

> **📜 Then (book, ~2020):** `python:3.7` / `python:3.11`  
> **🆕 Now (2026):** `python:3.12-slim` / `3.13-slim`  
> **Why:** Active security patches & fast wheel installs

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
A teammate pushed a syntax error straight to `main`, and manual FTP/SSH deploys left staging and production out of sync. How do you guarantee every commit is syntax-checked, tested, and bundled into the exact same artifact before deploy? You build fail-fast CI gates and automated promotion/rollback checks.

### Concept: Fail-Fast CI Gate
**What:** Automated lint/syntax/test stages  
**Why:** Blocks broken commits in seconds  
**How:** Non-zero exit code halts pipeline immediately  
**When:** Every repo on every push/PR

📖 *Book listing:* NFSOPS Jenkins `rsync` Job (pp. 259–262)

**Plain-language analogy:** A fail-fast CI gate is like an airport security scanner: if a bag fails inspection (syntax or test error), it stops on the belt immediately before boarding (production).

**Walkthrough:** The book uses Jenkins CI to run tests and `rsync -az --delete` code onto a shared Amazon EFS mount. Preconfigured Spot Instances read the updated code in under a second.

### Concept: Immutable Artifact Promotion
**What:** Build once, deploy everywhere  
**Why:** Prevents 'worked in staging' drift  
**How:** Bundle via `shutil.make_archive()`, promote tarball/image  
**When:** Staging -> Production releases

📖 *Book listing:* WordPress-to-Hugo Static Converter (pp. 238–248)

**Plain-language analogy:** Promoting the same artifact is like sealing a tested shipping container in staging and moving that exact container—not rebuilding a new one—into production.

**Walkthrough:** The book reads a WordPress JSON export, writes Hugo Markdown files with `+++` TOML frontmatter, copies images with `shutil`, and syncs `public/` to S3 + CloudFront.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: WordPress JSON -> Hugo Markdown converter (Book Case Study 1)
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: WordPress JSON -> Hugo Markdown converter (Book Case Study 1)
import shutil, tempfile
from pathlib import Path

posts = [{'slug': 'devops-101', 'title': 'DevOps 101', 'date': '2026-01-15', 'body': 'Automate CI/CD.'}]
with tempfile.TemporaryDirectory() as tmp:
    blog_dir = Path(tmp) / 'content' / 'blog'
    blog_dir.mkdir(parents=True)
    for p in posts:
        md_txt = f'+++\ntitle = "{p["title"]}"\ndate = "{p["date"]}"\n+++\n\n{p["body"]}\n'
        (blog_dir / f"{p['slug']}.md").write_text(md_txt, encoding='utf-8')
    print('Generated:', [f.name for f in blog_dir.iterdir()])

Generated: ['devops-101.md']


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: Fail-fast CI stage runner
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: Fail-fast CI stage runner
import subprocess

def run_ci_stage(name: str, cmd: list[str]) -> bool:
    cp = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
    print(f'[CI:{name}]', 'PASS' if cp.returncode == 0 else 'FAIL')
    return cp.returncode == 0

run_ci_stage('unit-smoke', ['python3', '-c', 'assert 2 + 2 == 4'])

[CI:unit-smoke] PASS


True

**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: Bundle release artifact with shutil.make_archive
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: Bundle release artifact with shutil.make_archive
import shutil, tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    src = Path(tmp) / 'app'
    src.mkdir()
    (src / 'main.py').write_text('print("ok")', encoding='utf-8')
    tar_path = shutil.make_archive(str(Path(tmp) / 'release-v1'), 'gztar', src)
    print('Artifact:', Path(tar_path).name)

Artifact: release-v1.tar.gz


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 7. 🟡 Intermediate Examples: Syntax Gate, GitHub Actions YAML & Flask App

#### Example 4: Intermediate Example 1: Recursive Python syntax gate via py_compile
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: Recursive Python syntax gate via py_compile
import py_compile, tempfile
from pathlib import Path

def syntax_check_dir(root: Path) -> dict[str, bool]:
    results: dict[str, bool] = {}
    for py in sorted(root.rglob('*.py')):
        try:
            py_compile.compile(str(py), doraise=True)
            results[py.name] = True
        except py_compile.PyCompileError:
            results[py.name] = False
    return results

with tempfile.TemporaryDirectory() as tmp:
    (Path(tmp) / 'good.py').write_text('x = 1\n')
    (Path(tmp) / 'bad.py').write_text('def broken(:\n')
    print('Syntax gate:', syntax_check_dir(Path(tmp)))

Syntax gate: {'bad.py': False, 'good.py': True}


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

#### Example 5: Intermediate Example 2 (External + Mock Fallback): Flask App Engine route
*Expected output:* Simulated fallback if optional package is missing.

In [5]:
# Intermediate Example 2 (External + Mock Fallback): Flask App Engine route
# Requires: pip install flask
try:
    from flask import Flask, jsonify
    app = Flask(__name__)
    @app.route('/health')
    def health(): return jsonify({'status': 'ok'})
    print('Flask health route registered')
except ImportError:
    def health(): return {'status': 'ok'}
    print('Flask not installed; simulated /health:', health())

Flask not installed; simulated /health: {'status': 'ok'}


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **Automated Canary Rollback Evaluator**—inspects post-deploy error rate and p95 latency to decide `PROMOTE` vs. `ROLLBACK`.

#### Example 6: def evaluate_canary(error_rate_pct: float, p95_ms: float, max_err: float = 1.0, max_p95: float = 300.0) -> str:
*Expected output:* Printed labelled output below.

In [6]:
def evaluate_canary(error_rate_pct: float, p95_ms: float, max_err: float = 1.0, max_p95: float = 300.0) -> str:
    return 'ROLLBACK' if (error_rate_pct > max_err or p95_ms > max_p95) else 'PROMOTE'

print('Healthy canary:', evaluate_canary(0.2, 110.0), '| Unhealthy:', evaluate_canary(2.4, 150.0))

Healthy canary: PROMOTE | Unhealthy: ROLLBACK


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Real Tool Demo (guarded with `shutil.which`)
*Expected output:* CLI version or check output (or graceful skip).

In [7]:
# Real Tool Demo: git & yamllint check on sample GitHub Actions YAML
import shutil, subprocess
if shutil.which('git'):
    print('CI git:', subprocess.run(['git', '--version'], capture_output=True, text=True, check=True).stdout.strip())
else:
    print('git not installed; skipped')

CI git: git version 2.56.1


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Swallowing non-zero exit codes** -> Broken builds deploy as green. ✅ Propagate `returncode != 0`.
- ❌ **Rebuilding per environment** -> Env-specific dependency drift. ✅ Promote the exact CI artifact.
- ❌ **Hardcoding cloud keys in CI** -> Credential leak in logs/forks. ✅ Use OIDC roles or repo secrets.

#### Example 7: def ci_exit_code(results: dict[str, bool]) -> int:
*Expected output:* Printed labelled output below.

In [8]:
def ci_exit_code(results: dict[str, bool]) -> int:
    return 0 if results and all(results.values()) else 1

print('Exit code when lint fails:', ci_exit_code({'lint': False, 'tests': True}))

Exit code when lint fails: 1


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 10. 🛠️ Hands-On Exercises
1. **(Beginner)** `slugify(title: str) -> str`: `'Hello DevOps!'` -> `'hello-devops'`.
2. **(Beginner)** `generate_algolia_index(posts: list[dict]) -> str`: JSON string of `{objectID, title, date}`.
3. **(Beginner)** `ci_exit_code(results: dict[str, bool]) -> int`: Return `0` iff all pass.
4. **(Intermediate)** `run_pipeline(stages: list[tuple[str, list[str]]]) -> bool`: Stop on first failing stage.
5. **(Intermediate)** `parse_gh_matrix(yaml_str: str) -> list[str]`: Extract Python versions from a GitHub Actions matrix (using `yaml.safe_load` or fallback).

## 11. 🏆 Practical DevOps Challenge
`ci_orchestrator(project_dir: Path) -> dict`: Run `syntax_check_dir()`, build `.tar.gz` only if all pass, return `{'passed': bool, 'checked_files': int, 'artifact': str | None}`.

> **🧪 Try it:** The starter cell below reports `TODO` until you implement the function body.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement slugify and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [9]:
# Starter Exercise Cell (implement slugify and run to verify!)
def _run_starter_checks():
    assert slugify('Hello DevOps!') == 'hello-devops'
def slugify(title: str) -> str:
    raise NotImplementedError('Implement slugify')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement slugify


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** How do Continuous Delivery and Continuous Deployment differ?

<details>
<summary>✅ Answer 1</summary>

Delivery keeps a manual approval gate before production; Deployment automates production release.

</details>

**Q2:** Why must failing CI steps return a non-zero exit code?

<details>
<summary>✅ Answer 2</summary>

CI runners halt pipelines and block merges/deploys on non-zero exit codes.

</details>

**Q3:** What does `aws s3 sync public/ s3://bucket --delete` do?

<details>
<summary>✅ Answer 3</summary>

Uploads changed files and deletes stale S3 objects removed from `public/`.

</details>

**Q4:** How does NFSOPS update thousands of Spot Instances in under a second?

<details>
<summary>✅ Answer 4</summary>

Jenkins `rsync`s code to a shared EFS/NFS mount attached to all nodes.

</details>

**Q5:** Why promote the exact same CI artifact from staging to production?

<details>
<summary>✅ Answer 5</summary>

Guarantees production runs the identical tested bits.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 1**: `re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')`.
- **Challenge**: Only call `shutil.make_archive` when `passed` is `True`.

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [10]:
# Complete Solutions + Assert Verification
import json, re, shutil, subprocess, tempfile
from pathlib import Path

def slugify(title: str) -> str:
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')

def generate_algolia_index(posts: list[dict]) -> str:
    return json.dumps([{'objectID': i + 1, 'title': p['title'], 'date': p['date']} for i, p in enumerate(posts)])

def run_pipeline(stages: list[tuple[str, list[str]]]) -> bool:
    for name, cmd in stages:
        if not run_ci_stage(name, cmd):
            return False
    return True

def ci_orchestrator(project_dir: Path) -> dict:
    checks = syntax_check_dir(project_dir)
    passed = bool(checks) and all(checks.values())
    artifact = shutil.make_archive(str(project_dir / 'artifact'), 'gztar', project_dir) if passed else None
    return {'passed': passed, 'checked_files': len(checks), 'artifact': artifact}

assert slugify('Hello DevOps!') == 'hello-devops'
assert json.loads(generate_algolia_index([{'title': 'A', 'date': '2026'}]))[0]['objectID'] == 1
assert run_pipeline([('s1', ['python3', '-c', 'print(1)'])]) is True
with tempfile.TemporaryDirectory() as tmp:
    (Path(tmp) / 'ok.py').write_text('x = 1\n')
    res = ci_orchestrator(Path(tmp))
    assert res['passed'] is True and res['artifact'] is not None
print('All Chapter 6 assertions PASSED!')

[CI:s1] PASS
All Chapter 6 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `py_compile.compile(p, doraise=True)` — Fast syntax check (e.g. `syntax_check_dir(Path('.'))`)
- `shutil.make_archive(b, 'gztar', d)` — Build release tarball (e.g. `release-v1.tar.gz`)
- `evaluate_canary(err, p95)` — Automated rollback gate (e.g. `'PROMOTE'` vs `'ROLLBACK'`)

### ✅ What You Can Now Do
- Build fail-fast CI runners using `py_compile`, `subprocess.run`, and `shutil.make_archive()`.
- Convert static content (WordPress -> Hugo) and generate search index JSON.
- Evaluate canary error rate and latency metrics to automate `PROMOTE` vs. `ROLLBACK`.

### ➡️ Next Chapter
Chapter 7 builds on this by adding structured logging and Prometheus/OpenTelemetry observability to deployed services.